In [0]:
"""
AeroPulse Enterprise Lakehouse Platform.

Synthetic maintenance event generator.

This module simulates maintenance events produced
by an operational maintenance application.
"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F


def generate_maintenance_events(
    spark: SparkSession,
    record_count: int = 5000,
) -> DataFrame:
    """
    Generate synthetic aircraft and engine maintenance events.

    Parameters
    ----------
    spark:
        Active SparkSession.
    record_count:
        Number of maintenance events to generate.

    Returns
    -------
    DataFrame
        Synthetic maintenance event data.
    """

    df = (
        spark.range(1, record_count + 1)

        .withColumn(
            "maintenance_event_id",
            F.format_string(
                "ME%010d",
                F.col("id")
            )
        )

        .withColumn(
            "aircraft_id",
            F.format_string(
                "AC%08d",
                ((F.col("id") - 1) % 1000) + 1
            )
        )

        .withColumn(
            "engine_id",
            F.format_string(
                "EN%09d",
                ((F.col("id") - 1) % 2000) + 1
            )
        )

        .withColumn(
            "maintenance_type",
            F.when(
                F.col("id") % 5 == 0,
                F.lit("ENGINE_INSPECTION")
            )
            .when(
                F.col("id") % 5 == 1,
                F.lit("ENGINE_REPAIR")
            )
            .when(
                F.col("id") % 5 == 2,
                F.lit("ROUTINE_INSPECTION")
            )
            .when(
                F.col("id") % 5 == 3,
                F.lit("COMPONENT_REPLACEMENT")
            )
            .otherwise(
                F.lit("UNSCHEDULED_REPAIR")
            )
        )

        .withColumn(
            "maintenance_status",
            F.when(
                F.col("id") % 10 == 0,
                F.lit("IN_PROGRESS")
            )
            .when(
                F.col("id") % 17 == 0,
                F.lit("CANCELLED")
            )
            .otherwise(
                F.lit("COMPLETED")
            )
        )

        .withColumn(
            "severity",
            F.when(
                F.col("id") % 20 == 0,
                F.lit("CRITICAL")
            )
            .when(
                F.col("id") % 7 == 0,
                F.lit("HIGH")
            )
            .when(
                F.col("id") % 3 == 0,
                F.lit("MEDIUM")
            )
            .otherwise(
                F.lit("LOW")
            )
        )

        .withColumn(
            "maintenance_location",
            F.when(
                F.col("id") % 4 == 0,
                F.lit("LONDON")
            )
            .when(
                F.col("id") % 4 == 1,
                F.lit("SINGAPORE")
            )
            .when(
                F.col("id") % 4 == 2,
                F.lit("DUBAI")
            )
            .otherwise(
                F.lit("FRANKFURT")
            )
        )

        .withColumn(
            "technician_id",
            F.format_string(
                "TECH%06d",
                ((F.col("id") - 1) % 500) + 1
            )
        )

        .withColumn(
            "maintenance_cost_usd",
            (
                F.lit(500)
                + (F.col("id") % 50) * 125
            ).cast("decimal(12,2)")
        )

        .withColumn(
            "event_timestamp",
            F.current_timestamp()
        )

        .drop("id")
    )

    return df